# CNN & Pooling

> **Cheat Sheet Sections covered:** ⑨ Convolution (CNN) · ⑩ Pooling · ③ Network Architectures (CNN)

Convolutional Neural Networks are the backbone of computer vision. Instead of connecting every input to every neuron (like a dense layer), a **convolutional layer** slides a small filter across the input, detecting local patterns — edges, textures, shapes.

```mermaid
flowchart LR
    A["Input Image
(H × W × C)"] --> B["Conv Layer
Filters / Kernels"]
    B --> C["Feature Maps
Activation (ReLU)"]
    C --> D["Pooling Layer
Max / Avg Pool"]
    D --> E["Flatten"]
    E --> F["Dense / MLP
Classifier"]
```


## 1. Convolution Operation

A **filter** (kernel) is a small matrix of learned weights that slides across the input:
- **Stride**: how many pixels to move the filter each step
- **Padding**: add zeros around the border to control output size
- **Output size**: $\lfloor (W - F + 2P) / S \rfloor + 1$

Each filter detects one type of feature. Multiple filters → multiple **feature maps**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

# ── Manual convolution to build intuition ─────────────────────────────────────
def manual_conv2d(image, kernel, stride=1, padding=0):
    """Apply a 2D convolution manually (no padding/batching for clarity)"""
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    output = np.zeros((out_H, out_W))
    for i in range(0, out_H):
        for j in range(0, out_W):
            region = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            output[i, j] = np.sum(region * kernel)
    return output

# Create a simple 8x8 "image"
img = np.array([
    [0,0,0,0,0,0,0,0],
    [0,1,1,1,0,0,0,0],
    [0,1,0,1,0,0,0,0],
    [0,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,0,0],
    [0,0,0,0,1,0,0,0],
    [0,0,0,0,1,1,0,0],
    [0,0,0,0,0,0,0,0],
], dtype=float)

# Edge-detection kernels
kernels = {
    "Horizontal Edge": np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=float),
    "Vertical Edge":   np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float),
    "Sharpen":         np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original Image', fontsize=12); axes[0].axis('off')

for ax, (name, k) in zip(axes[1:], kernels.items()):
    out = manual_conv2d(img, k)
    ax.imshow(out, cmap='RdBu', interpolation='nearest')
    ax.set_title(f'Filter: {name}', fontsize=11); ax.axis('off')

plt.suptitle('Convolution: What Different Filters Detect', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("Each filter extracts a different spatial feature from the same image.")


## 2. PyTorch Conv2d

`nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)` is the standard building block.

| Parameter | Meaning |
|---|---|
| `in_channels` | Number of input feature maps (1 for grayscale, 3 for RGB) |
| `out_channels` | Number of filters = number of output feature maps |
| `kernel_size` | Filter size (e.g. 3 = 3×3) |
| `stride` | Step size (default 1) |
| `padding` | Zero-padding (use `padding='same'` to keep spatial size) |


In [ ]:
# PyTorch Conv2d shapes demo
batch = torch.randn(8, 1, 28, 28)   # 8 grayscale 28×28 images
print(f"Input:  {tuple(batch.shape)}  (batch, channels, H, W)")

conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
out1  = conv1(batch)
print(f"After Conv2d(1→32, k=3, p=1): {tuple(out1.shape)}")

conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
out2  = conv2(F.relu(out1))
print(f"After Conv2d(32→64, k=3, p=1): {tuple(out2.shape)}")

print(f"\nconv1 learnable params: {sum(p.numel() for p in conv1.parameters()):,}")
print(f"  weight: {tuple(conv1.weight.shape)}  (out_ch, in_ch, kH, kW)")
print(f"  bias:   {tuple(conv1.bias.shape)}")
print(f"\nA dense layer connecting 28×28 input to 32×28×28 would need: "
      f"{28*28 * 32*28*28:,} params")
print(f"Conv2d only needs: {sum(p.numel() for p in conv1.parameters()):,}  ← parameter sharing!")


## 3. Pooling: Max & Average

Pooling **downsamples** feature maps, reducing spatial size while retaining the strongest signal.

| Type | Operation | Use |
|---|---|---|
| **Max Pooling** | Takes the maximum value in each window | Preserves strongest feature response |
| **Average Pooling** | Takes the mean value in each window | Smoother, used in global pooling |
| **Global Average Pooling** | Average over entire spatial dimension | Replace flattening before classifier |


In [ ]:
# Visualise Max vs Average Pooling
feature_map = torch.tensor([[
    [1., 3., 2., 4.],
    [5., 6., 0., 1.],
    [2., 1., 3., 2.],
    [4., 0., 1., 5.],
]]).unsqueeze(0)  # (1, 1, 4, 4)

max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

mp_out = max_pool(feature_map)
ap_out = avg_pool(feature_map)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes,
        [feature_map[0,0], mp_out[0,0], ap_out[0,0]],
        ['Input Feature Map (4×4)', 'Max Pool 2×2 → (2×2)', 'Avg Pool 2×2 → (2×2)']):
    im = ax.imshow(data.numpy(), cmap='YlOrRd', vmin=0, vmax=6)
    for r in range(data.shape[0]):
        for col in range(data.shape[1]):
            ax.text(col, r, f'{data[r,col]:.0f}', ha='center', va='center',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontsize=12); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Max Pooling vs Average Pooling', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Input shape:    {tuple(feature_map.shape)}")
print(f"MaxPool output: {tuple(mp_out.shape)}")
print(f"AvgPool output: {tuple(ap_out.shape)}")


## 4. A Complete CNN for MNIST

Building a full convolutional architecture:  
`Conv → BN → ReLU → MaxPool → Conv → BN → ReLU → MaxPool → Flatten → Dense → Output`


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,),(0.3081,))])
train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_ld = DataLoader(train_ds, batch_size=128, shuffle=True)
test_ld  = DataLoader(test_ds,  batch_size=1000, shuffle=False)

class SimpleCNN(nn.Module):
    """
    Classic Conv → Pool → Conv → Pool → FC architecture.
    Input: (B, 1, 28, 28) grayscale image
    Output: (B, 10) logits
    """
    def __init__(self):
        super().__init__()
        # Block 1: 1→16 feature maps, 28×28 → 14×14
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2: 16→32 feature maps, 14×14 → 7×7
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Classifier
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.drop  = nn.Dropout(0.3)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # → (B,16,14,14)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # → (B,32,7,7)
        x = x.view(x.size(0), -1)                         # flatten
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)

model = SimpleCNN()
total = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {total:,}")

# Shape trace
dummy = torch.randn(1, 1, 28, 28)
print(f"\nShape trace:")
print(f"  Input:          {tuple(dummy.shape)}")
x = model.pool1(F.relu(model.bn1(model.conv1(dummy))))
print(f"  After Block 1:  {tuple(x.shape)}")
x = model.pool2(F.relu(model.bn2(model.conv2(x))))
print(f"  After Block 2:  {tuple(x.shape)}")
x = x.view(x.size(0), -1)
print(f"  After Flatten:  {tuple(x.shape)}")


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss/len(loader), 100.*correct/total

print("Training CNN on MNIST (5 epochs)...\n")
for ep in range(5):
    tr_loss, tr_acc = run_epoch(train_ld, train=True)
    te_loss, te_acc = run_epoch(test_ld,  train=False)
    print(f"Epoch {ep+1}/5 | Train {tr_acc:.1f}% | Test {te_acc:.1f}% | "
          f"Loss {tr_loss:.4f}")

print("\n✅ CNN training complete!")


## 5. Visualising What the CNN Learns

After training, we can look at the filters in the first layer — the patterns the network has learned to detect.


In [ ]:
# Visualise learned conv1 filters
filters = model.conv1.weight.data.cpu()  # (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
axes = axes.ravel()
for i in range(16):
    axes[i].imshow(filters[i, 0], cmap='RdBu', interpolation='nearest')
    axes[i].set_title(f'F{i}', fontsize=9); axes[i].axis('off')
plt.suptitle('Learned Conv1 Filters (3×3)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("Each filter detects a different low-level pattern (edges, blobs, corners).")


## Summary

### ✅ What You Learned

1. **Convolution** slides filters over inputs to extract local features — far fewer parameters than dense layers
2. **Max Pooling** retains the strongest signal; **Average Pooling** smooths
3. **BatchNorm2d** stabilises CNN training
4. Stacking Conv→Pool blocks builds increasingly abstract representations
5. CNNs reach >99% on MNIST with under 100k parameters

### 🎯 What's Next

- **13_lstm** — sequence modelling with gates and cell state
- **06_attention_mechanism** — attention replaces recurrence for sequences
